In [3]:
!pip install mistralai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.1/261.1 kB 8.2 MB/s eta 0:00:00


In [4]:
import os
from google.cloud import storage
import time
import random
import re
import json
import os
from mistralai import Mistral
from vertexai.generative_models import Part


In [5]:
api_key = os.environ["MISTRAL_API_KEY"]  # set your own Mistral API key


In [6]:


client = Mistral(api_key=api_key)
content = '''
Genre: Romantic travel vlog.
Sound: Yes.

[My boyfriend woke me up at 4 am for a surprise Valentine's Day trip]
A YOUNG WOMAN is looking out of the airplane's window while wearing wireless headphones. She looks sleepy. She turns to look back at her BOYFRIEND.
A BOYFRIEND is loading bags into a black car.
The couple is driving.
BOYFRIEND is driving the car. He is wearing a black cap and sunglasses.
The car is now driving near the red rocks and canyons of Arizona.
[He found the most stunning hotel in Sedona, Arizona- AMBIENTE LANDSCAPE HOTEL]
The couple is hugging while taking a selfie.
The view from a luxurious hotel of the red rocks of Sedona, Arizona.
A couple is taking a selfie near the hotel, BOYFRIEND puts his arm around the YOUNG WOMAN. They are both wearing hotel robes.
[They have an amazing spa]
The couple's room has a great view of the mountains and some flowers on the table.
The hotel's SPA.
[Hiking trails from their property]
A blue buggy parked near a trail.
The couple is taking a selfie near a sign, making a heart shape with their hands.
BOYFRIEND is smiling at the camera.
A heart-shaped cactus on the trail.
BOYFRIEND is walking down the trail.
[And the most incredible on-site restaurant]
Plates of gourmet food from the hotel's restaurant.
A heart-shaped dessert.
'''

messages = [
    {
        "role": "user",
        "content": content,
    }
]
chat_response = client.agents.complete(
    agent_id="ag:28a0913a:20250114:annotation-bro:0464a208",

      messages = messages,
      response_format = {
          "type": "json_object",
      }
)

print(chat_response.choices[0].message.content)


{
  "1_Value1_Self_Direction_Thought_values": "None",
  "1_Value1_Self_Direction_Action_values": "present",
  "1_Value1_Stimulation_values": "present",
  "1_Value1_Hedonism_values": "present",
  "1_Value1_Achievement_values": "None",
  "1_Value1_Power_Resources_values": "present",
  "1_Value1_Power_dominance_values": "conflict",
  "1_Value1_Face_values": "None",
  "1_Value1_Security_Personal_values": "None",
  "1_Value1_Security_Social_values": "None",
  "1_Value1_Conformity_Rules_values": "None",
  "1_Value1_Conformity_Interpersonal_values": "present",
  "1_Value1_Tradition_values": "None",
  "1_Value1_Humility_values": "None",
  "1_Value1_Benevolence_Dependability_values": "None",
  "1_Value1_Benevolence_Care_values": "present",
  "1_Value1_Universalism_Concern_values": "None",
  "1_Value1_Universalism_Nature_values": "present",
  "1_Value1_Universalism_Tolerance_values": "None",
  "Has_sound": "True"
}


In [8]:
from google.cloud import storage

def read_gcs_file(uri: str) -> str:
    """
    Reads a file from Google Cloud Storage using the provided URI.

    Args:
        uri (str): The URI of the file in Google Cloud Storage (e.g., gs://influencers_tiktok/movies/videos/VideoScripts/@mongraalreal_video_7361025467631144225.mp4).

    Returns:
        str: The content of the file.
    """
    if not uri.startswith("gs://"):
        raise ValueError("The URI must start with 'gs://'")

    # Remove the 'gs://' prefix
    uri = uri[5:]

    # Split the URI into bucket name and blob path
    parts = uri.split("/", 1)

    if len(parts) < 2:
        raise ValueError("Invalid URI: Missing blob path.")

    bucket_name = parts[0]
    blob_name = parts[1]

    # Initialize the Google Cloud Storage client
    storage_client = storage.Client()

    # Get the bucket and blob objects
    bucket = storage_client.get_bucket(bucket_name)
    blob = bucket.blob(blob_name)

    # Read the content of the blob (file)
    content = blob.download_as_text()

    return content

# Example usage:
uri = "gs://influencers_tiktok/movies/videos/VideoScripts/@mongraalreal_video_7361025467631144225.mp4"
try:
    file_content = read_gcs_file(uri)
    print("File content:")
    print(file_content)
except Exception as e:
    print(f"Error: {e}")


File content:
Genre: Gaming/Comedy
Sound: Yes

STREAMER A: "Go on Mongraal stream and look what he just did. We'll be laughing for weeks. Wasn't even that bad, bro. It was actually indescribable how bad it was. His whole chat is gonna laugh endlessly."
[GO ON]
[AND LOOK WHAT]
[HE JUST DID]
[FOR WEEKS]
[THAT BAD]
[UNDESCRIBABLE]
[HOW SH*T IT]
[HIS WHOLE]
[IT'S GONNA]
[ENDLESSLY]
STREAMER A is watching Mongraal's live stream on Twitch, he is excited about something Mongraal did.
STREAMER A: "Here it comes."
[HERE IT COMES]
STREAMER A is fully focused on the stream, he is watching Mongraal's Fortnite gameplay.
STREAMER A: "Seven shots to kill that guy, and now now comes the fun."
[TO KILL]
[THAT GUY]
[COMES THE FUN]
Mongraal gets hit by something and makes a noise.
STREAMER A starts laughing.
STREAMER A: "Wasn't even that bad."
[THAT BAD]
STREAMER A is laughing hysterically, while Mongraal looks confused.
STREAMER A: "Bro."
Mongraal is looking for heals.
STREAMER A: "I should go. I should

In [9]:
def list_all_moviescripts(bucket_name, source_blob_name):
    """Lists all txt blobs in the bucket.

    Args:
        bucket_name: The name of the Google Cloud Storage bucket.
        source_blob_name: The path prefix to look for within the bucket.
    """

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix=source_blob_name)
    blob_uris = []

    for blob in blobs:

        if  blob.content_type=="text/plain":  # Filter for MP4 files
            blob_uri = f"gs://{bucket_name}/{blob.name}"
            blob_uris.append(blob_uri)
        else:
            print(f"Skipping {blob.name} not an txt file.")

    return blob_uris



In [10]:
def extract_json_and_text(input_string):
    """
    Extract JSON and text content from a string containing Markdown code blocks.

    Parameters:
        input_string (str): The input string with ```json and ```text blocks.

    Returns:
        tuple: A tuple containing (json_dict, text_str).
               json_dict is the parsed JSON as a dictionary and the extracted text in a 'Additional Notes' field.
    """
    # Regex patterns for JSON and text blocks
    json_pattern = r'```json\s*(.*?)\s*```'
    text_pattern = r'```text\s*(.*?)\s*```'

    # Extract JSON and text components
    json_match = re.search(json_pattern, input_string, re.DOTALL)
    text_match = re.search(text_pattern, input_string, re.DOTALL)

    # Initialize the outputs
    json_dict = None
    text_str = None

    # Parse JSON if found
    if json_match:
        try:
            json_dict = json.loads(json_match.group(1))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

    # Extract plain text if found
    if text_match:
        text_str = text_match.group(1).strip()

    # Add the text to the JSON dictionary
    if text_str is not None:
      json_dict["Additional Notes"] = text_str

    return json_dict


In [11]:
# ----- Function to generate annotation from a single video -----
def generate_annotation_from_script(
    script_uri: str, max_retries=4, retry_delay=40
) -> str:
    """Generates an annotation from a single video URI using Gemini 1.5 Pro.

    Args:
        script_uri: The GCS URI of the text file.
        max_retries: Maximum number of retries if a request fails
        retry_delay: Base delay (in seconds) between retries

    Returns:
        a JSON string containing the generated annotation.
        a TXT file containing additional notes
    """
    content = read_gcs_file(script_uri)

    messages = [
        {
            "role": "user",
          "content": content,
        }
        ]

    for attempt in range(max_retries):
      try:
        print("trying " + script_uri)
        response = client.agents.complete(
            agent_id="ag:28a0913a:20250114:annotation-bro:0464a208",
            messages = messages,
            response_format = {
                "type": "json_object",
                }
)

        if response.choices[0].message.content:

            return json.loads(response.choices[0].message.content)
        else:
            return f"Error: Could not generate annotations for {script_uri}, {response}"

      except Exception as e:
        print(f"Error generating annotations for {script_uri} (attempt {attempt + 1}): {e}")
        if attempt < max_retries - 1:
           delay = (retry_delay * (2**attempt)) + random.uniform(0, 1)
           print(f"Retrying in {delay:.2f} seconds...")
           time.sleep(delay)
        else:
            return f"Error: Failed to generate annotations for {script_uri} after {max_retries} retries. Error:{response.prompt_feedback}"

    return f"Error: Failed to generate annotations for {script_uri} after {max_retries} retries. Error:{response.prompt_feedback}"

In [12]:
import csv
from typing import List, Tuple, Dict, Any
from google.cloud import storage
import io
import os
import urllib.parse


def create_csv_from_uris_gcs(uris: List[str], bucket_name: str, blob_path: str):
    """
    Fetches data from URIs using the 'generate_annotation_from_script' function and saves the CSV to GCS
    at the specified relative path.

    Args:
        uris: A list of URIs to process.
        bucket_name: The name of the GCS bucket.
        blob_path: The desired path for the CSV file in GCS, including any relative directories.
    """

    all_keys = ['filename']  # Start with filename as the first key
    failures = []

    # Initialize GCS client
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_path)

    # Create a file-like object for writing to a blob
    with blob.open("w", content_type="text/csv") as file_like_object:
        writer = None  # initialize the writer to None

        for uri in uris:
            try:
                data = generate_annotation_from_script(uri)
                if type(data) == str:
                    print(data)
                    failures.append(data)
                    continue
            except Exception as e:
                print(f"Error generating annotations for {uri}: {e}")
                continue

            # Extract filename from URI
            filename = uri.split('/')[-1]
            filename = filename[:-4] if filename.endswith(('.mp4', '.avi')) else filename

            # Add filename to the data dictionary
            try:
                data['filename'] = filename
            except Exception as e:
                print(f"Error adding filename for {uri}: {e}")
                print(data)
                continue



            # Add new keys while preserving order
            for key in data.keys():
                if key not in all_keys:
                    all_keys.append(key)

            if not writer:  # if writer was not created, it means it's the first row, so we create it and write the headers
                writer = csv.DictWriter(file_like_object, fieldnames=all_keys)
                writer.writeheader()

            writer.writerow(data)
            print(f"Processed {uri}")

    print(f"CSV file uploaded to gs://{bucket_name}/{blob_path}")
    return failures


In [13]:
# unnote these when ready to go over all videos
# bucket_name = 'influencers_tiktok'
source_blob_name = "movies/videos/VideoScripts"
# OUTPUT_DIRECTORY = "VideoScripts"

bucket_name = 'influencers_tiktok'
# source_blob_name = "movies/test_space/test_scripts/"
destination_folder = "movies/test_space"
filename = "scripted_annotations_mistral.csv"
final_path = os.path.join(destination_folder, filename)

# list URIs
uris = list_all_moviescripts(bucket_name, source_blob_name)

# # run the functions
logs = create_csv_from_uris_gcs(uris[1:], bucket_name, final_path)
# logs = create_csv_from_uris_gcs(['gs://influencers_tiktok/movies/videos/VideoScripts/@avani_video_7343015745980337438.mp4'], bucket_name, final_path)

trying gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6783398367490854150.mp4
Processed gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6783398367490854150.mp4
trying gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6807140917636648197.mp4
Processed gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6807140917636648197.mp4
trying gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6811970678024195334.mp4
Processed gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6811970678024195334.mp4
trying gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6905879944979631365.mp4
Processed gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6905879944979631365.mp4
trying gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6925220555478027525.mp4
Processed gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6925220555478027525.mp4
trying gs://inf